In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# ==========================================================
# FULL CODE WITH IMPROVED FUSION
# ==========================================================

import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from sklearn.metrics import confusion_matrix, classification_report
from tqdm import tqdm
from copy import deepcopy
import matplotlib.pyplot as plt
import seaborn as sns
import timm

# ==========================================================
# PATHS
# ==========================================================

BASE_SAVE_DIR = "./dpGCViT_new_Results"
FIG_DIR = os.path.join(BASE_SAVE_DIR, "figures")
MODEL_DIR = os.path.join(BASE_SAVE_DIR, "models")

os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

MODEL_PATH = os.path.join(MODEL_DIR, "best_model.pth")

DATA_ROOT     = "/content/drive/MyDrive/The Oxford-IIIT Pet Dataset"
IMAGES_DIR    = os.path.join(DATA_ROOT, "images", "images")
ANNOT_DIR     = os.path.join(DATA_ROOT, "annotations", "annotations")

TRAINVAL_FILE = os.path.join(ANNOT_DIR, "trainval.txt")
TEST_FILE     = os.path.join(ANNOT_DIR, "test.txt")

# ==========================================================
# CONFIGURATION
# ==========================================================

IMG_SIZE = 224
BATCH_SIZE = 8
ACCUM_STEPS = 2
EPOCHS = 100
EMA_DECAY = 0.999
MODEL_TYPE = "dp_gcvit"
PATIENCE = 10
PRINT_ALPHA_FREQ = 50
NUM_WORKERS = 0
INIT_ALPHA = 0.5
AUX_LOSS_WEIGHT = 0.2
FUSION_MODE = 'adaptive'  # 'gated' | 'feature' | 'residual' | 'adaptive'

# ==========================================================
# SEED
# ==========================================================

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
print(f"Model: {MODEL_TYPE}")
print(f"Fusion Mode: {FUSION_MODE}")

# ==========================================================
# CLASSES
# ==========================================================

ALL_CLASSES = sorted(list(set([line.strip().split()[0].rsplit("_", 1)[0]
                                for line in open(TRAINVAL_FILE)])))
class_to_idx = {c: i for i, c in enumerate(ALL_CLASSES)}
num_classes = len(ALL_CLASSES)
print(f"Number of classes: {num_classes}")

# ==========================================================
# DATA TRANSFORMS
# ==========================================================

train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(0.3, 0.3, 0.3, 0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.4)
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# ==========================================================
# DATASET
# ==========================================================

class OxfordPetDataset(Dataset):
    def __init__(self, list_file, img_dir, transform=None):
        self.transform = transform
        self.img_dir = img_dir
        self.samples = []
        with open(list_file) as f:
            for line in f:
                name = line.strip().split()[0]
                breed = name.rsplit("_", 1)[0]
                self.samples.append((name + ".jpg", class_to_idx[breed]))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_name, label = self.samples[idx]
        image = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label

# ==========================================================
# DATALOADERS
# ==========================================================

full_dataset = OxfordPetDataset(TRAINVAL_FILE, IMAGES_DIR)
indices = list(range(len(full_dataset)))
random.shuffle(indices)
split = int(0.8 * len(indices))
train_idx, val_idx = indices[:split], indices[split:]

train_dataset = torch.utils.data.Subset(
    OxfordPetDataset(TRAINVAL_FILE, IMAGES_DIR, train_transform), train_idx
)
val_dataset = torch.utils.data.Subset(
    OxfordPetDataset(TRAINVAL_FILE, IMAGES_DIR, eval_transform), val_idx
)
test_dataset = OxfordPetDataset(TEST_FILE, IMAGES_DIR, eval_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")

# ==========================================================
# DP_GCViT WITH IMPROVED FUSION
# ==========================================================

class GCViT_Baseline(nn.Module):
    def __init__(self, backbone, num_classes):
        super().__init__()
        self.backbone = backbone
        self.num_features = backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(self.num_features),
            nn.Dropout(0.3),
            nn.Linear(self.num_features, num_classes)
        )

    def forward(self, x):
        feat = self.backbone.forward_features(x)
        if feat.dim() == 4:
            feat = feat.flatten(2).transpose(1, 2)
        global_feat = feat.mean(dim=1)
        return self.head(global_feat)


class DP_GCViT(nn.Module):
    def __init__(self, backbone, num_classes, init_alpha=0.5, fusion_mode='feature'):
        super().__init__()
        self.backbone = backbone
        self.num_features = backbone.num_features
        self.init_alpha = init_alpha
        self.fusion_mode = fusion_mode

        # Shared projection
        self.proj = nn.Sequential(
            nn.LayerNorm(self.num_features),
            nn.Linear(self.num_features, self.num_features),
            nn.GELU(),
            nn.Dropout(0.1)
        )

        # Separate classifiers
        self.part_fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.num_features, num_classes)
        )
        self.global_fc = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(self.num_features, num_classes)
        )

        # ===== IMPROVED FUSION =====
        # Gate for weight computation
        self.fusion_gate = nn.Sequential(
            nn.Linear(self.num_features * 2, self.num_features // 2),
            nn.GELU(),
            nn.Dropout(0.1),
            nn.Linear(self.num_features // 2, 2),
        )

        # Feature-level fusion classifier
        self.fusion_fc = nn.Sequential(
            nn.Linear(self.num_features * 2, self.num_features),
            nn.LayerNorm(self.num_features),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(self.num_features, num_classes)
        )

        # Residual fusion projection
        self.residual_proj = nn.Linear(self.num_features, self.num_features)

        # Adaptive fusion (confidence-based)
        self.confidence_proj = nn.Linear(num_classes, 1)

        self._init_fusion_gate()

    def _init_fusion_gate(self):
        with torch.no_grad():
            self.fusion_gate[-1].bias.data.fill_(0.0)
            for module in self.fusion_gate.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight, gain=0.5)
            # Initialize fusion_fc
            for module in self.fusion_fc.modules():
                if isinstance(module, nn.Linear):
                    nn.init.xavier_uniform_(module.weight, gain=0.5)
            # Initialize residual_proj
            nn.init.xavier_uniform_(self.residual_proj.weight, gain=0.5)
            nn.init.zeros_(self.residual_proj.bias)

    def forward(self, x, return_all=False):
        feat = self.backbone.forward_features(x)
        if feat.dim() == 4:
            feat = feat.flatten(2).transpose(1, 2)

        # Extract features
        global_feat = feat.mean(dim=1)
        part_feat = feat.max(dim=1)[0]

        # Project
        global_feat = self.proj(global_feat)
        part_feat = self.proj(part_feat)

        # Individual outputs
        out_global = self.global_fc(global_feat)
        out_part = self.part_fc(part_feat)

        # ===== FUSION STRATEGIES =====
        if self.fusion_mode == 'gated':
            # Standard gated fusion (original)
            gate_input = torch.cat([global_feat, part_feat], dim=1)
            fusion_weights = torch.softmax(self.fusion_gate(gate_input), dim=1)
            alpha = fusion_weights[:, 1:2]
            out_fused = fusion_weights[:, 0:1] * out_global + fusion_weights[:, 1:2] * out_part

        elif self.fusion_mode == 'feature':
            # Feature-level fusion (BEST)
            combined = torch.cat([global_feat, part_feat], dim=1)
            out_fused = self.fusion_fc(combined)
            # Compute alpha for analysis
            gate_input = torch.cat([global_feat, part_feat], dim=1)
            alpha = torch.softmax(self.fusion_gate(gate_input), dim=1)[:, 1:2]

        elif self.fusion_mode == 'residual':
            # Residual fusion: learn what part features add
            residual = self.residual_proj(part_feat - global_feat)
            fused_feat = global_feat + 0.5 * residual  # 0.5 to prevent overfitting
            out_fused = self.global_fc(fused_feat)
            gate_input = torch.cat([global_feat, part_feat], dim=1)
            alpha = torch.softmax(self.fusion_gate(gate_input), dim=1)[:, 1:2]

        elif self.fusion_mode == 'adaptive':
            # Adaptive confidence-based fusion
            probs_global = torch.softmax(out_global, dim=1)
            probs_part = torch.softmax(out_part, dim=1)

            # Compute confidence (max probability)
            conf_global, _ = torch.max(probs_global, dim=1, keepdim=True)
            conf_part, _ = torch.max(probs_part, dim=1, keepdim=True)

            # Weight based on relative confidence
            total_conf = conf_global + conf_part + 1e-8
            weight_global = conf_global / total_conf
            weight_part = conf_part / total_conf

            out_fused = weight_global * out_global + weight_part * out_part
            alpha = weight_part.squeeze(-1)

        else:
            # Default: simple average
            out_fused = (out_global + out_part) / 2
            alpha = torch.ones(out_global.size(0), 1, device=x.device) * 0.5
            alpha = alpha.squeeze(-1)

        if return_all:
            return out_fused, out_global, out_part, alpha.squeeze(-1) if alpha.dim() > 1 else alpha
        return out_fused


# ==========================================================
# CREATE MODEL
# ==========================================================

backbone = timm.create_model("gcvit_tiny", pretrained=True, num_classes=0)
model = DP_GCViT(backbone, num_classes, init_alpha=INIT_ALPHA, fusion_mode=FUSION_MODE)
model = model.to(device)
ema_model = deepcopy(model).to(device)
ema_model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

# ==========================================================
# OPTIMIZER, SCHEDULER, LOSS
# ==========================================================

optimizer = optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.05)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = GradScaler()

# ==========================================================
# TRAINING FUNCTIONS
# ==========================================================

def train_epoch(model, loader, optimizer, criterion, scaler, device, epoch, ema_model, aux_weight=0.2):
    model.train()
    running_loss = 0
    correct_fused, correct_global, correct_part = 0, 0, 0
    total = 0
    epoch_alphas = []

    for i, (x, y) in enumerate(tqdm(loader, desc=f"Epoch {epoch+1} [Train]")):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

        with autocast():
            out_fused, out_global, out_part, alpha = model(x, return_all=True)

            loss_fused = criterion(out_fused, y)
            loss_global = criterion(out_global, y)
            loss_part = criterion(out_part, y)

            # Main loss with auxiliary losses
            loss = loss_fused + aux_weight * (loss_global + loss_part)
            loss = loss / ACCUM_STEPS

        epoch_alphas.append(alpha.mean().item())
        scaler.scale(loss).backward()

        if (i + 1) % ACCUM_STEPS == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()

            with torch.no_grad():
                for ema_p, p in zip(ema_model.parameters(), model.parameters()):
                    ema_p.data.mul_(EMA_DECAY).add_(p.data, alpha=1 - EMA_DECAY)

        running_loss += loss.item() * ACCUM_STEPS

        pred_fused = out_fused.argmax(1)
        pred_global = out_global.argmax(1)
        pred_part = out_part.argmax(1)

        correct_fused += (pred_fused == y).sum().item()
        correct_global += (pred_global == y).sum().item()
        correct_part += (pred_part == y).sum().item()
        total += y.size(0)

        if (i + 1) % PRINT_ALPHA_FREQ == 0:
            print(f"  Batch {i+1}: Alpha = {alpha.mean().item():.4f} ± {alpha.std().item():.4f}")

    train_loss = running_loss / len(loader)
    train_acc_fused = 100 * correct_fused / total
    train_acc_global = 100 * correct_global / total
    train_acc_part = 100 * correct_part / total
    avg_alpha = np.mean(epoch_alphas)

    return train_loss, train_acc_fused, train_acc_global, train_acc_part, avg_alpha


def validate(model, loader, criterion, device):
    model.eval()
    val_loss = 0
    correct_fused, correct_global, correct_part = 0, 0, 0
    total = 0
    val_alphas = []

    with torch.no_grad():
        for x, y in tqdm(loader, desc="Validating"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            out_fused, out_global, out_part, alpha = model(x, return_all=True)
            loss = criterion(out_fused, y)

            val_loss += loss.item()
            val_alphas.append(alpha.mean().item())

            pred_fused = out_fused.argmax(1)
            pred_global = out_global.argmax(1)
            pred_part = out_part.argmax(1)

            correct_fused += (pred_fused == y).sum().item()
            correct_global += (pred_global == y).sum().item()
            correct_part += (pred_part == y).sum().item()
            total += y.size(0)

    val_loss /= len(loader)
    val_acc_fused = 100 * correct_fused / total
    val_acc_global = 100 * correct_global / total
    val_acc_part = 100 * correct_part / total
    avg_alpha = np.mean(val_alphas)

    return val_loss, val_acc_fused, val_acc_global, val_acc_part, avg_alpha


# ==========================================================
# MAIN TRAINING LOOP
# ==========================================================

if __name__ == '__main__':

    train_loss_history = []
    train_acc_fused_history = []
    train_acc_global_history = []
    train_acc_part_history = []
    val_acc_fused_history = []
    val_acc_global_history = []
    val_acc_part_history = []
    val_loss_history = []
    alpha_history = []
    lr_history = []

    best_acc = 0
    patience_counter = 0

    print("\n" + "=" * 70)
    print("STARTING TRAINING")
    print("=" * 70)
    print(f"Fusion Mode: {FUSION_MODE}")
    print(f"Auxiliary Loss Weight: {AUX_LOSS_WEIGHT}")
    print("=" * 70)

    for epoch in range(EPOCHS):
        train_loss, train_fused, train_global, train_part, train_alpha = train_epoch(
            model, train_loader, optimizer, criterion, scaler, device, epoch, ema_model, AUX_LOSS_WEIGHT
        )

        val_loss, val_fused, val_global, val_part, val_alpha = validate(
            ema_model, val_loader, criterion, device
        )

        train_loss_history.append(train_loss)
        val_loss_history.append(val_loss)
        train_acc_fused_history.append(train_fused)
        train_acc_global_history.append(train_global)
        train_acc_part_history.append(train_part)
        val_acc_fused_history.append(val_fused)
        val_acc_global_history.append(val_global)
        val_acc_part_history.append(val_part)
        alpha_history.append(val_alpha)
        lr_history.append(scheduler.get_last_lr()[0])

        scheduler.step()

        print(f"\n{'=' * 70}")
        print(f"EPOCH {epoch + 1}/{EPOCHS}")
        print(f"{'=' * 70}")
        print(f"TRAIN - Fused: {train_fused:.2f}% | Global: {train_global:.2f}% | Part: {train_part:.2f}%")
        print(f"VALID - Fused: {val_fused:.2f}% | Global: {val_global:.2f}% | Part: {val_part:.2f}%")
        print(f"Alpha: {val_alpha:.4f} | LR: {lr_history[-1]:.6f}")

        if val_fused > best_acc:
            best_acc = val_fused
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': ema_model.state_dict(),
                'best_acc': best_acc,
                'fusion_mode': FUSION_MODE,
            }, MODEL_PATH)
            print(f"\n>>> SAVED BEST MODEL! Validation Fused: {best_acc:.2f}% <<<\n")
        else:
            patience_counter += 1
            print(f"No improvement ({patience_counter}/{PATIENCE})")
            if patience_counter >= PATIENCE:
                print("\n>>> EARLY STOPPING TRIGGERED! <<<\n")
                break

    # Load best model
    checkpoint = torch.load(MODEL_PATH)
    ema_model.load_state_dict(checkpoint['model_state_dict'])
    print(f"\nLoaded best model from epoch {checkpoint['epoch'] + 1}")
    print(f"Best validation fused accuracy: {checkpoint['best_acc']:.2f}%")

    # ==========================================================
    # TESTING
    # ==========================================================

    ema_model.eval()
    correct_fused, correct_global, correct_part = 0, 0, 0
    total = 0
    y_true, y_pred_fused, y_pred_global, y_pred_part = [], [], [], []
    test_alphas = []

    print("\n" + "=" * 70)
    print("TESTING WITH TTA")
    print("=" * 70)

    with torch.no_grad():
        for x, y in tqdm(test_loader, desc="Testing"):
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)

            out_fused1, out_global1, out_part1, alpha1 = ema_model(x, return_all=True)
            out_fused2, out_global2, out_part2, alpha2 = ema_model(torch.flip(x, [3]), return_all=True)

            out_fused = (out_fused1 + out_fused2) / 2
            out_global = (out_global1 + out_global2) / 2
            out_part = (out_part1 + out_part2) / 2
            alpha = (alpha1 + alpha2) / 2

            pred_fused = out_fused.argmax(1)
            pred_global = out_global.argmax(1)
            pred_part = out_part.argmax(1)

            correct_fused += (pred_fused == y).sum().item()
            correct_global += (pred_global == y).sum().item()
            correct_part += (pred_part == y).sum().item()
            total += y.size(0)

            y_true.extend(y.cpu().numpy())
            y_pred_fused.extend(pred_fused.cpu().numpy())
            y_pred_global.extend(pred_global.cpu().numpy())
            y_pred_part.extend(pred_part.cpu().numpy())
            test_alphas.extend(alpha.cpu().numpy().flatten())

    test_acc_fused = 100 * correct_fused / total
    test_acc_global = 100 * correct_global / total
    test_acc_part = 100 * correct_part / total

    print(f"\n{'=' * 50}")
    print("FINAL TEST RESULTS")
    print(f"{'=' * 50}")
    print(f"FUSED Path (alpha-weighted):  {test_acc_fused:.2f}%")
    print(f"GLOBAL Path (avg pooling):    {test_acc_global:.2f}%")
    print(f"PART Path (max pooling):      {test_acc_part:.2f}%")
    print(f"{'=' * 50}")
    print(f"Alpha Statistics on Test Set:")
    print(f"  Mean:  {np.mean(test_alphas):.4f}")
    print(f"  Std:   {np.std(test_alphas):.4f}")
    print(f"  Min:   {np.min(test_alphas):.4f}")
    print(f"  Max:   {np.max(test_alphas):.4f}")
    print(f"{'=' * 50}")

    # ==========================================================
    # PLOT RESULTS
    # ==========================================================

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    # Training accuracy
    axes[0, 0].plot(train_acc_fused_history, label='Fused', linewidth=2, color='green')
    axes[0, 0].plot(train_acc_global_history, label='Global', linewidth=2, color='blue')
    axes[0, 0].plot(train_acc_part_history, label='Part', linewidth=2, color='red')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy (%)')
    axes[0, 0].set_title('Training Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Validation accuracy
    axes[0, 1].plot(val_acc_fused_history, label='Fused', linewidth=2, color='green')
    axes[0, 1].plot(val_acc_global_history, label='Global', linewidth=2, color='blue')
    axes[0, 1].plot(val_acc_part_history, label='Part', linewidth=2, color='red')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Accuracy (%)')
    axes[0, 1].set_title('Validation Accuracy')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Best validation comparison
    x_pos = [0, 1, 2]
    best_val_fused = max(val_acc_fused_history)
    best_val_global = max(val_acc_global_history)
    best_val_part = max(val_acc_part_history)
    heights = [best_val_fused, best_val_global, best_val_part]
    colors = ['green', 'blue', 'red']
    bars = axes[0, 2].bar(x_pos, heights, color=colors, alpha=0.7, edgecolor='black')
    axes[0, 2].set_xticks(x_pos)
    axes[0, 2].set_xticklabels(['Fused', 'Global', 'Part'])
    axes[0, 2].set_ylabel('Accuracy (%)')
    axes[0, 2].set_title('Best Validation Accuracy')
    axes[0, 2].set_ylim(0, 100)
    for bar, h in zip(bars, heights):
        axes[0, 2].text(bar.get_x() + bar.get_width()/2, h + 1, f'{h:.1f}%', ha='center', fontweight='bold')
    axes[0, 2].grid(True, alpha=0.3, axis='y')

    # Add epoch information for best accuracy
    epoch_fused = val_acc_fused_history.index(best_val_fused) + 1
    epoch_global = val_acc_global_history.index(best_val_global) + 1
    epoch_part = val_acc_part_history.index(best_val_part) + 1
    axes[0, 2].text(0, -5, f'Epoch {epoch_fused}', ha='center', fontsize=9, style='italic', color='green')
    axes[0, 2].text(1, -5, f'Epoch {epoch_global}', ha='center', fontsize=9, style='italic', color='blue')
    axes[0, 2].text(2, -5, f'Epoch {epoch_part}', ha='center', fontsize=9, style='italic', color='red')

    # Alpha curve
    axes[1, 0].plot(alpha_history, color='purple', linewidth=2)
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('Alpha Value')
    axes[1, 0].set_title('Gate Alpha (Part Feature Weight)')
    axes[1, 0].set_ylim(0, 1)
    axes[1, 0].axhline(y=0.5, color='gray', linestyle='--', label='Balanced')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Alpha distribution
    axes[1, 1].hist(test_alphas, bins=50, color='purple', alpha=0.7, edgecolor='black')
    axes[1, 1].set_xlabel('Alpha Value')
    axes[1, 1].set_ylabel('Frequency')
    axes[1, 1].set_title('Alpha Distribution on Test Set')
    axes[1, 1].axvline(x=0.5, color='red', linestyle='--', label='Alpha=0.5')
    axes[1, 1].axvline(x=np.mean(test_alphas), color='orange', linestyle='--', label=f'Mean: {np.mean(test_alphas):.3f}')
    axes[1, 1].legend()
    axes[1, 1].grid(True, alpha=0.3)

    # Learning rate
    axes[1, 2].plot(lr_history, color='brown', linewidth=2)
    axes[1, 2].set_xlabel('Epoch')
    axes[1, 2].set_ylabel('Learning Rate')
    axes[1, 2].set_title('Learning Rate Schedule')
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, "training_analysis.png"), dpi=300)
    plt.show()

    # ==========================================================
    # SUMMARY
    # ==========================================================

    print("\n" + "=" * 80)
    print("TRAINING COMPLETED!")
    print("=" * 80)
    print(f"Fusion Mode: {FUSION_MODE}")
    print(f"Best validation FUSED accuracy: {best_acc:.2f}%")
    print(f"\nFinal Test Results:")
    print(f"  FUSED Path:  {test_acc_fused:.2f}%")
    print(f"  GLOBAL Path: {test_acc_global:.2f}%")
    print(f"  PART Path:   {test_acc_part:.2f}%")
    print(f"\nAlpha Analysis:")
    print(f"  Test mean alpha: {np.mean(test_alphas):.4f}")
    print(f"  Test alpha std:  {np.std(test_alphas):.4f}")

    if test_acc_fused > test_acc_global and test_acc_fused > test_acc_part:
        print("\n✅ FUSION IMPROVED ACCURACY!")
    elif test_acc_fused >= test_acc_global and test_acc_fused >= test_acc_part:
        print("\n⚠️ FUSION MATCHED BEST PATH")
    else:
        print("\n❌ FUSION DID NOT IMPROVE - Try different fusion mode")

    print(f"\nResults saved to: {BASE_SAVE_DIR}")
    print("=" * 80)

Using device: cpu
Model: dp_gcvit
Fusion Mode: gated
Number of classes: 37
Train: 2944, Val: 736, Test: 3669
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights-morevit/gcvit_tiny_224_nvidia-ac783954.pth" to /root/.cache/torch/hub/checkpoints/gcvit_tiny_224_nvidia-ac783954.pth


/tmp/ipykernel_1949/3173958097.py:335: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/grad_scaler.py:31: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  super().__init__(


Total parameters: 29,081,029
Trainable parameters: 29,081,029

STARTING TRAINING
Fusion Mode: gated
Auxiliary Loss Weight: 0.2


Epoch 1 [Train]:   0%|          | 0/368 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/tmp/ipykernel_1949/3173958097.py:351: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
/usr/local/lib/python3.12/dist-packages/torch/cuda/amp/autocast_mode.py:54: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  super().__init__(
Epoch 1 [Train]:  14%|█▎        | 50/368 [06:55<41:19,  7.80s/it]

  Batch 50: Alpha = 0.0342 ± 0.0478


...

Epoch 1 [Train]:  41%|████      | 150/368 [20:10<30:03,  8.27s/it]

  Batch 150: Alpha = 0.0000 ± 0.0000


Epoch 1 [Train]:  54%|█████▍    | 200/368 [26:39<22:26,  8.01s/it]

  Batch 200: Alpha = 0.0000 ± 0.0000


Epoch 1 [Train]:  60%|██████    | 221/368 [29:31<20:36,  8.41s/it]